In [1]:
import torch
from dinosaw.linear_probe import get_ramp, gen_sample_mask, do_linear_probe, LinearProbeResult
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models
from dinosaw.utils import do_2D_pca, get_features

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False
torch.cuda.empty_cache()

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s',)
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

2026-08-11 14:08:16 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-08-11 14:08:16 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-08-11 14:08:16 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-08-11 14:08:16 | I | factory.py                 : 152 | Building wrapper 'dinov3_s+' on device cuda:0
2026-08-11 14:08:16 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/

In [3]:
SF = 0.5

img_fname = "../../tests/default_image.jpg"

_img = Image.open(f"{img_fname}").convert("RGB")
_img = _img.resize((int(SF * _img.width), int(SF * _img.height)), Image.BILINEAR)

In [4]:
ramp_type = 'lr'
chosen_vis_model = 'dinov3_s+'

results: dict[ModelTypes, LinearProbeResult] = {}
feats_reduced: dict[str, np.ndarray] = {}


In [5]:
for model_key in enabled_models:
    model = models[model_key]
    feats = get_features(model, _img, True)
    probe_res = do_linear_probe(feats, ramp_type, probe_by_channel=True, mask_cutoff_frac=0.8)

    results[model_key] = probe_res
    feats_reduced[model_key] = do_2D_pca(feats.transpose((2, 0, 1)), 3, pre_norm='std', post_norm='minmax')

2026-08-11 14:09:09 | I | wrapper.py                 :  92 | Processing image, size: [512, 448]
2026-08-11 14:09:09 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,448,504] -> f: [1,384,32,36]
2026-08-11 14:09:09 | I | wrapper.py                 :  92 | Processing image, size: [512, 448]
2026-08-11 14:09:09 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,448,512] -> f: [1,384,28,32]
2026-08-11 14:09:10 | I | wrapper.py                 :  92 | Processing image, size: [512, 448]
2026-08-11 14:09:10 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,448,504] -> f: [1,384,32,36]


In [6]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)


def set_axes_off(ax: plt.Axes):
    ax.set_xticks([])
    ax.set_yticks([])

In [13]:
# %%capture
FS = 24
S = 14

fig = plt.figure(figsize=(20, 20))
gs = GridSpec(1 + len(enabled_models), 3, wspace=0.07, hspace=0.15, width_ratios=[1, 1, 1])


img_ax = fig.add_subplot(gs[0,0])
img_ax.imshow(_img)
set_axes_off(img_ax)
img_ax.set_title("Image", fontsize=FS)

pca_ax = fig.add_subplot(gs[0, 1])
pca_ax.imshow(feats_reduced[chosen_vis_model])
set_axes_off(pca_ax)
pca_ax.set_title(f"Feats ({MODEL_NAMES[chosen_vis_model]})", fontsize=FS)


h ,w = (_img.height // S, _img.width // S)
ramp = get_ramp(ramp_type, h, w)
mask = gen_sample_mask((h, w), ramp_type, cutoff_frac=0.8)

ramp_ax = fig.add_subplot(gs[0, 2])
ramp_ax.imshow(ramp)
set_axes_off(ramp_ax)
ramp_ax.set_title("Target ramp ()", fontsize=FS)

add_red_square_overlay(pca_ax, mask, 1,  1)
add_red_square_overlay(ramp_ax, mask, 1,  1)


colors = {'dv2': ''}

for i, model_key in enumerate(enabled_models):
    probe_res = results[model_key]

    scores_ax = fig.add_subplot(gs[i + 1, 0:2])
    scores_ax.set_ylabel(f"{MODEL_NAMES[model_key]}", fontsize=FS)
    scores_ax.plot(probe_res['per_channel_scores'], lw=2)
    scores_ax.set_ylim(-0.3, 1)
    scores_ax.tick_params(axis='both', which='major', labelsize=FS - 8)

    if i == 0:
        scores_ax.set_title('Per channel ' +"$R^{2}$", fontsize=FS)
    if i == len(enabled_models) - 1:
        scores_ax.set_xlabel('Channel', fontsize=FS)
        # scores_ax.set_ylabel("$R^{2}:$")

    pred_ax = fig.add_subplot(gs[i + 1, 2])
    pred_ax.imshow(probe_res['stack_pred'])
    pred_ax.set_ylabel(r"$R^{2}: $" + f"{probe_res['stack_r_squared']:.3f}", fontsize=FS)
    set_axes_off(pred_ax)


SHOW = False
if SHOW:
    plt.show()
else:
    plt.close()